In [1]:
import pricing_driven_service_allocation as pdsa
# URLS
PRIME_INSTANCE_URL = "http://localhost:3000/api/v1/"
# Paths
TOPOLOGIES_DIR = "synthetic-dataset/synthetic-topologies"
DEVICES_EUA_DATASET_PATH = "eua-dataset/edge-servers/site.csv"
CLIENTS_EUA_DATASET_PATH = "eua-dataset/users/users-aus.csv"
EXPERIMENT_CONFIGURATION_PATH = "experiments/experiment_test.yml"
# Scenario containers
topologies = {}
demands = {}
filters = {}
results = {}
# Configuration
SEED = 42
UNLIMITED_VALUE = 100000000
VENDORS_TO_CONSIDER = ["Telstra", "Optus", "Vodafone", "Telecom", "Macquarie"]
OFFER_CONFIGURATION = {
    "global": {
        "group_percentages": {1: 40, 2: 30, 3: 30},
        "group_ranges": {1: (0, 12.5), 2: (12.5, 50), 3: (50, 100)},
    },
    "attributes": {
        "available_ram_gb": {
            "min": 1,
            "max": 128,
            "default_price": 0.5,
            "price_by_provider_group": {
                "OPTUS": {1: 1, 2: 0.15, 3: 0.005},
                "TELSTRA": {1: 1.3, 2: 0.12, 3: 0.003},
                "VODAFONE": {1: 1.5, 2: 0.18, 3: 0.001},
                "MACQUARIE": {1: 0.2, 2: 0.17, 3: 0.008},
                "TELECOM": {1: 0.7, 2: 0.14, 3: 0.004},
            },
            "local_distribution": {
                1: [(70, 0, 25), (30, 25, 100)]
            },
        },

        "available_storage_gb": {
            "min": 1,
            "max": 2000,  # gigabytes; adjust if you want very large storage nodes
            "default_price": 0.2,
            "price_by_provider_group": {
                "OPTUS": {1: 0.90, 2: 0.25, 3: 0.08},
                "TELSTRA": {1: 1.10, 2: 0.30, 3: 0.10},
                "VODAFONE": {1: 1.25, 2: 0.35, 3: 0.05},
                "MACQUARIE": {1: 0.50, 2: 0.20, 3: 0.09},
                "TELECOM": {1: 0.85, 2: 0.28, 3: 0.07},
            },
            "local_distribution": {
                # Conserva tu intención: muchos dispositivos con poco almacenamiento y unos pocos con más
                1: [(70, 0, 2), (30, 2, 25)]
            },
        },

        "available_cpu_cores": {
            "min": 1,
            "max": 64,
            "default_price": 15,
        },

        # Unidades equivalentes, coherentes con gpu_equivalent_units en demanda
        "available_gpu_units": {
            "min": 0,
            "max": 8,
            "default_price": 120,
            "price_by_provider_group": {
                "TELSTRA": {2: 130, 3: 150},
                "OPTUS": {3: 140},
            },
            "local_distribution": {
                1: [(85, 0, 25), (15, 25, 50)],
                2: [(60, 0, 50), (40, 50, 100)],
                3: [(30, 0, 50), (70, 50, 100)],
            },
        },

        "available_tpu_units": {
            "min": 0,
            "max": 4,
            "default_price": 200,
            "price_by_provider_group": {
                "TELSTRA": {3: 240},
                "OPTUS": {2: 210},
            },
            "local_distribution": {
                1: [(90, 0, 25), (10, 25, 50)],
                2: [(70, 0, 50), (30, 50, 100)],
                3: [(40, 0, 50), (60, 50, 100)],
            },
        },

        # Opcional: si luego quieres que el algoritmo también restrinja por ancho de banda
        "available_network_in_mbps": {
            "min": 10,
            "max": 10000,
            "default_price": 0.01,
            "local_distribution": {
                1: [(80, 0, 20), (20, 20, 100)],
                2: [(60, 20, 60), (40, 60, 100)],
                3: [(20, 60, 100), (80, 80, 100)],
            },
        },
        "available_network_out_mbps": {
            "min": 10,
            "max": 10000,
            "default_price": 0.01,
            "local_distribution": {
                1: [(80, 0, 20), (20, 20, 100)],
                2: [(60, 20, 60), (40, 60, 100)],
                3: [(20, 60, 100), (80, 80, 100)],
            },
        },
    },

    "device_types_by_group": {
        1: {"SENSOR": 25, "CAMERA": 50, "MOBILE": 25},
        2: {"LAPTOP": 25, "COMPUTER": 50, "MOBILE": 25},
        3: {"DATA_CENTER": 75, "SERVER": 25},
    },
}
RESOURCES_TO_CONSIDER = [key for key in OFFER_CONFIGURATION['attributes'].keys() if 'network' not in key]

In [2]:
import yaml
import os

# Load the YAML file
with open(os.path.join(EXPERIMENT_CONFIGURATION_PATH), 'r') as file:
  config_data = yaml.safe_load(file)
  
for scenario_size in config_data['experiments'].keys():
    for app in config_data['experiments'][scenario_size].keys():
        topology_offer = config_data['experiments'][scenario_size][app]['offer']
        # Check that all providers in the topology offer are in the vendors to consider
        providers_in_offer = [p.lower() for p in topology_offer['providers'].keys()]
        vendors_lower = [v.lower() for v in VENDORS_TO_CONSIDER]

        if not all(provider in vendors_lower for provider in providers_in_offer):
          raise ValueError(f"Some providers in topology {scenario_size}/{app} offer are not in VENDORS_TO_CONSIDER."
                   f"Providers in offer: {providers_in_offer}, "
                   f"Vendors to consider: {vendors_lower}")

In [3]:
import os
import shutil

# Create all defined paths if they do not exist
paths_to_create = [
  TOPOLOGIES_DIR,
]

for path in paths_to_create:
  if path and not os.path.exists(path):
    os.makedirs(path, exist_ok=True)
    print(f"Created directory: {path}")
  elif path and os.path.exists(path):
    print(f"Directory already exists: {path}. Removing existing subfolders...")
    # Remove existing subfolders (including non-empty ones)
    for subfolder in os.listdir(path):
      subfolder_path = os.path.join(path, subfolder)
      if os.path.isdir(subfolder_path):
        shutil.rmtree(subfolder_path)
    print("Done!")
    

Directory already exists: synthetic-dataset/synthetic-topologies. Removing existing subfolders...
Done!


# 2. Filter EUA dataset

In this step, we will filter the EUA dataset to include only devices from which we can infer the vendor, usually throug their description. This is crucial for our analysis as it allows us to categorize devices and simulate exclusion and interoperability relationships among providers.

In addition, we will enrich the dataset with synthetic resources, unit prices, and device types. This enrichment is based on the OFFER_CONFIGURATION defined at the beginning of the notebook.

In [4]:
# Load dataset
devices_df = pdsa.dataset.load_devices_dataframe(DEVICES_EUA_DATASET_PATH)
# Filter devices from which vendor can be inferred
devices_df = pdsa.dataset.filter_devices_by_vendors(devices_df, VENDORS_TO_CONSIDER)
# Assign resources to devices based on the offer configuration
devices_df = pdsa.dataset.assign_device_resources(devices_df, OFFER_CONFIGURATION, SEED)

devices_df.head()

,latitude,longitude,elevation,provider,global_group,device_type,available_ram_gb,unit_price_available_ram_gb,available_storage_gb,unit_price_available_storage_gb,available_cpu_cores,unit_price_available_cpu_cores,available_gpu_units,unit_price_available_gpu_units,available_tpu_units,unit_price_available_tpu_units,available_network_in_mbps,unit_price_available_network_in_mbps,available_network_out_mbps,unit_price_available_network_out_mbps
device_id,,,,,,,,,,,,,,,,,,,,
10000002,-28.777660,114.634260,NaN,OPTUS,1,CAMERA,5,1.000,5,0.90,4,15.0,0,120.0,0,200.0,25,0.01,983,0.01
100001,-38.248652,144.605442,23.0,TELSTRA,1,MOBILE,12,1.300,9,1.10,6,15.0,0,120.0,0,200.0,130,0.01,239,0.01
10000114,-31.901910,152.533540,NaN,OPTUS,3,DATA_CENTER,95,0.005,1818,0.08,39,15.0,7,140.0,3,200.0,9296,0.01,9484,0.01
100002,-37.728550,145.222007,116.0,OPTUS,2,MOBILE,61,0.150,394,0.25,29,15.0,1,120.0,0,210.0,4437,0.01,2391,0.01
10000215,-32.981570,121.644400,NaN,TELSTRA,2,MOBILE,22,0.120,949,0.30,11,15.0,1,130.0,1,200.0,2270,0.01,2359,0.01


# 3. Topology Generation

In [5]:

for scenario_size in config_data['experiments'].keys():
    for app in config_data['experiments'][scenario_size].keys():
        topology_offer = config_data['experiments'][scenario_size][app]['offer']
        _, topology_id = pdsa.generators.topology(
                          lat = topology_offer['zone']['lat'],
                          long = topology_offer['zone']['long'],
                          center_elevation = topology_offer['zone'].get('elevation', 0),
                          rad=topology_offer['zone']['radius'],
                          devices_df=devices_df,
                          topologies_result_dir=TOPOLOGIES_DIR,
                          resources_to_consider=RESOURCES_TO_CONSIDER,
                          number_of_providers=len(topology_offer["providers"]),
                          allowed_groups=[1, 2, 3],
                          number_of_devices=topology_offer['max_devices'],
                          options={"seed": SEED, "logs": False}
                      )
        topologies[f"{scenario_size}_{app}"] = topology_id
        
print("---- Topologies generated ----\n")
for key, value in topologies.items():
  print(f"{key}: {value}")

---- Topologies generated ----

small_ar/vr: 3d263016-c460-4292-ab5f-2d2cab94ee5d
medium_ar/vr: 407d54c5-a1c9-4b28-9815-7b35376cef5a
large_ar/vr: 3ceb4d8d-da8d-4ece-92d7-cd8f5a815c8a


# 4. Problem instance generation

In [6]:
import os

if not os.path.exists("./iPricing/model"):
    os.makedirs("./iPricing/model")

!protoc --python_out=./iPricing/model ./iPricing/iPricing.proto
!mv ./iPricing/model/iPricing/iPricing_pb2.py ./iPricing/model/iPricing_pb2.py
!rm -rf ./iPricing/model/iPricing

In [7]:
import pandas as pd
from pricing_driven_service_allocation.generators.client_demand import AppType

c_locations_df = pd.read_csv(os.path.join(CLIENTS_EUA_DATASET_PATH), index_col=0)
n_clients = len(c_locations_df)

app_mapping = {
  "ar/vr": AppType.AR_VR,
  "video_privacy": AppType.VIDEO_PRIVACY,
  "lidar": AppType.LIDAR,
  "robot_iot": AppType.ROBOT_IOT
}

In [8]:
from tqdm import tqdm
from iPricing.model.iPricing_pb2 import Pricing

for id, topology_id in tqdm(topologies.items(), desc="Generating instances"):
  scenario_size, app = id.split("_")
  topology_demand = config_data['experiments'][scenario_size][app]['demand']
  topology_request = config_data['experiments'][scenario_size][app]['request']
  
  users_demand = pdsa.generators.client_demand.calculate_resources(n_clients*topology_demand['clients_density'], 
                                                                   app_mapping[app], 
                                                                   concurrency=topology_demand['concurrency'], 
                                                                   requests_per_second_per_active_client=topology_demand['requests_per_second_per_active_client'], 
                                                                   requests_per_second_std=topology_demand['requests_per_second_std'], 
                                                                   resources_to_consider=RESOURCES_TO_CONSIDER, 
                                                                   random_state=SEED)
  
  request = pdsa.generators.request(
      topology_demand=topology_demand,
      topology_request=topology_request,
      users_demand=users_demand,
      resources_to_consider=RESOURCES_TO_CONSIDER,
  )
  
  print(request)
  
  pricing_path = pdsa.generators.pricing_from_topology(
            topology_id=topology_id,
            topologies_result_dir=TOPOLOGIES_DIR,
            resources_to_consider=RESOURCES_TO_CONSIDER,
            compatible_provider_groups=pdsa.generators.compatible_provider_groups_from_offer(topology_offer),
            options={"logs": False}
          )
  
  pricing_obj = pdsa.utils.yaml_to_pricing_proto(
      os.path.join(TOPOLOGIES_DIR, topology_id, "pricing.yml"), 
      Pricing
  )
  
  problem_instance_pricing, filter_criteria = pdsa.generators.problem_instance(
                                                  instance_pricing=pricing_obj,
                                                  request=request,
                                                  topologies_result_dir=TOPOLOGIES_DIR,
                                                  unlimited_value=UNLIMITED_VALUE
                                              )
  
  pdsa.utils.pricing_proto_to_yaml(
    problem_instance_pricing, 
    os.path.join(TOPOLOGIES_DIR, topology_id, "problem_instance_pricing.yml"),
    options={"logs": False}
  )
  
  filters[id] = filter_criteria
  

Generating instances:   0%|          | 0/3 [00:00<?, ?it/s]

{'currency': 'USD', 'users_location': [[144.95128085314013, -37.81311379425756], [144.954906094624, -37.82109949971801], [144.97481006469786, -37.81512407043976], [144.97090595848454, -37.807413124398344], [144.95128085314013, -37.81311379425756]], 'providers_to_consider': ['TELSTRA', 'OPTUS'], 'budget': 10000, 'max_devices': 2, 'device_types': ['CAMERA', 'DATA_CENTER'], 'resources': {'available_ram_gb': 12, 'available_storage_gb': 61, 'available_cpu_cores': 11, 'available_gpu_units': 1, 'available_tpu_units': 0}, 'max_distance': 25000}


Generating instances:  33%|███▎      | 1/3 [00:00<00:00,  4.54it/s]

{'currency': 'USD', 'users_location': [[144.95128085314013, -37.81311379425756], [144.954906094624, -37.82109949971801], [144.97481006469786, -37.81512407043976], [144.97090595848454, -37.807413124398344], [144.95128085314013, -37.81311379425756]], 'providers_to_consider': ['TELSTRA', 'OPTUS'], 'budget': 10000, 'max_devices': 5, 'device_types': ['CAMERA', 'DATA_CENTER'], 'resources': {'available_ram_gb': 18, 'available_storage_gb': 118, 'available_cpu_cores': 23, 'available_gpu_units': 1, 'available_tpu_units': 0}, 'max_distance': 25000}


Generating instances:  67%|██████▋   | 2/3 [00:11<00:06,  6.63s/it]

{'currency': 'USD', 'users_location': [[144.95128085314013, -37.81311379425756], [144.954906094624, -37.82109949971801], [144.97481006469786, -37.81512407043976], [144.97090595848454, -37.807413124398344], [144.95128085314013, -37.81311379425756]], 'providers_to_consider': ['TELSTRA', 'OPTUS'], 'budget': 10000, 'max_devices': 10, 'device_types': ['CAMERA', 'DATA_CENTER'], 'resources': {'available_ram_gb': 23, 'available_storage_gb': 175, 'available_cpu_cores': 11, 'available_gpu_units': 1, 'available_tpu_units': 0}, 'max_distance': 25000}


Generating instances: 100%|██████████| 3/3 [00:24<00:00,  8.20s/it]


# 5. Pricing Optimization

In [9]:
for id in topologies.keys():
  instance_path = os.path.join(TOPOLOGIES_DIR, topologies[id], "problem_instance_pricing.yml")
  
  print(topologies[id])
  
  filter = filters[id]
  
  result = pdsa.optimize(PRIME_INSTANCE_URL, instance_path, filter)
  
  results[id] = result
  
  break
  
  

3d263016-c460-4292-ab5f-2d2cab94ee5d
Submitting optimization job with filters:
{
  "maxPrice": 10000,
  "maxSubscriptionSize": 2,
  "usageLimits": {
    "available_ram_gb": 12,
    "available_storage_gb": 61,
    "available_cpu_cores": 11,
    "available_gpu_units": 1,
    "available_tpu_units": 0,
    "distance": 99975000
  },
  "features": [
    "CAMERA",
    "DATA_CENTER"
  ]
}


In [10]:
print(results["small_ar/vr"])

{'jobId': 'job-fac6fe6a-25bb-475c-a56f-ff6d577d2aa3', 'status': 'COMPLETED', 'submittedAt': '2026-02-17T12:45:42.092Z', 'startedAt': '2026-02-17T12:45:42.092Z', 'completedAt': '2026-02-17T12:45:42.304Z', 'result': {'optimal': {'subscription': {'addOns': ['OPTUS_134449', 'TELSTRA_37789'], 'features': ['CAMERA', 'DATA_CENTER'], 'usageLimits': [{'available_tpu_units': 3}, {'available_ram_gb': 71}, {'available_cpu_cores': 49}, {'distance': 199997488}, {'available_storage_gb': 1340}, {'available_gpu_units': 6}]}, 'cost': '346.74 $'}}}
